# 12. Prepare A2D2 — resume-safe v2

A2D2 `20190401_121727` Front Center + Bus를 Stage 3 v5용 10 Hz 데이터로 변환한다.

이 버전은 Colab 런타임 종료를 전제로 설계했다.
- TAR scan은 Google Drive checkpoint에서 재개
- index 완성 후에는 91.4 GiB TAR 전체 scan 재실행 없음
- 600-frame/60초 segment마다 checkpoint
- 완료 segment는 재실행 시 SKIP
- camera tar를 Google Drive에 풀지 않음
- GPU 불필요


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
BRANCH = 'stage3-sangchun'

if not REPO.exists():
    subprocess.run([
        'git','clone','-b',BRANCH,
        'https://github.com/sangchun1/Blackbox-Detection.git',
        str(REPO),
    ], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)

SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('python:', sys.version)
print('repo  :', REPO)
print('src   :', SRC)


Mounted at /content/drive
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
repo  : /content/Blackbox-Detection
src   : /content/Blackbox-Detection/src


## 1. 패치 파일 적용 확인

이 notebook과 함께 제공한 `src/blackbox_detection/stage3/a2d2.py`를 repo 같은 경로에 덮어쓰고 commit/push한 뒤 실행한다.


In [2]:
A2D2_MODULE = REPO / 'src/blackbox_detection/stage3/a2d2.py'
assert A2D2_MODULE.is_file(), A2D2_MODULE
print('a2d2.py:', A2D2_MODULE)
print('bytes  :', A2D2_MODULE.stat().st_size)


a2d2.py: /content/Blackbox-Detection/src/blackbox_detection/stage3/a2d2.py
bytes  : 19967


In [3]:
from blackbox_detection.stage3.a2d2 import (
    A2D2PrepareConfig,
    load_bus_json,
    audit_bus,
    prepare_session_resume,
)
print('A2D2 resume module: PASS')


A2D2 resume module: PASS


## 2. 데이터 경로 확인


In [4]:
A2D2_ROOT = DRIVE_ROOT / 'DATASET/A2D2'
ARCHIVE_DIR = A2D2_ROOT / 'archives'
RAW_ROOT = A2D2_ROOT / 'raw'
PROCESSED_ROOT = A2D2_ROOT / 'processed'
SESSION_ID = '20190401_121727'

CAMERA_TAR = ARCHIVE_DIR / 'camera_lidar-20190401121727_camera_frontcenter.tar'
BUS_JSON = RAW_ROOT / 'camera_lidar' / SESSION_ID / 'bus' / '20190401121727_bus_signals.json'

assert CAMERA_TAR.is_file(), CAMERA_TAR
assert BUS_JSON.is_file(), BUS_JSON
print('camera tar GiB:', CAMERA_TAR.stat().st_size / 1024**3)
print('bus json MiB  :', BUS_JSON.stat().st_size / 1024**2)


camera tar GiB: 91.40242576599121
bus json MiB  : 176.42605018615723


## 3. 첫 v2 실행 전에만 old v1 산출물 정리 (선택)

기존 run은 index checkpoint를 남기지 못했으므로 재사용할 수 있는 핵심 결과가 없다.
단, 원본 `archives/`와 `raw/.../bus/`는 절대 삭제하지 않는다.

**v2 checkpoint가 한 번이라도 생긴 뒤에는 이 cleanup 셀을 다시 실행하지 않는다.**


In [5]:
# 처음 v2로 갈아탈 때만 필요하면 실행.
# import shutil
# route = 'a2d2_20190401_121727'
# for p in [
#     PROCESSED_ROOT / 'videos' / route,
#     PROCESSED_ROOT / 'metadata' / route,
#     PROCESSED_ROOT / 'aux_metadata' / route,
#     PROCESSED_ROOT / 'done' / route,
#     PROCESSED_ROOT / 'cache' / route,
# ]:
#     if p.exists():
#         print('remove:', p)
#         shutil.rmtree(p)


## 4. Bus audit


In [6]:
bus = load_bus_json(BUS_JSON)
r = audit_bus(bus)
print(r)
assert r['accel_dvdt_corr'] > 0.80
assert r['steer_yaw_corr'] > 0.50
print('BUS AUDIT: PASS')


{'accel_dvdt_corr': 0.9700520460458929, 'steer_yaw_corr': 0.7271866055177758, 'accel_sign': '+acceleration_x', 'steering_sign_rule': 'sign=1 -> negative; sign=0 -> positive'}
BUS AUDIT: PASS


## 5. Resume-safe full preparation

런타임이 끊기면 새 런타임에서 1, 2, 4, 5만 다시 실행한다.
`overwrite=False`를 유지해야 resume가 작동한다.


In [7]:
import json
from pathlib import Path
cfg = A2D2PrepareConfig(
    processed_root=PROCESSED_ROOT,
    width=512,
    height=384,
    target_fps=10,
    segment_frames=600,
    crf=23,
    ffmpeg_preset='veryfast',
    max_camera_skew_ms=25.0,
    scan_checkpoint_every=500,
    overwrite=False,
    work_root=Path('/content/a2d2_work'),
)

report = prepare_session_resume(
    camera_tar=CAMERA_TAR,
    bus_json=BUS_JSON,
    session_id=SESSION_ID,
    cfg=cfg,
)
print(json.dumps(report, indent=2))


camera index: REUSE (27451 json, 27451 png)
[01/16] 000: SKIP
[02/16] 001: SKIP
[03/16] 002: SKIP
[04/16] 003: SKIP
[05/16] 004: SKIP
[06/16] 005: SKIP
[07/16] 006: SKIP
[08/16] 007: SKIP
[09/16] 008: SKIP
[10/16] 009: SKIP
[11/16] 010: SKIP
[12/16] 011: SKIP
[13/16] 012: SKIP
[14/16] 013: SKIP
[15/16] 014: SKIP
[16/16] 015: SKIP
{
  "session_id": "20190401_121727",
  "index_report": {
    "source": "final_index",
    "path": "/content/drive/MyDrive/Blackbox-Detection/DATASET/A2D2/processed/cache/a2d2_20190401_121727/camera_index.json"
  },
  "camera_records": 27451,
  "selected_10hz_frames": 9151,
  "segments": 16,
  "duration_hours": 0.25419444444444445,
  "camera_skew_ms": {
    "max_abs": 16.669,
    "p95_abs": 15.792
  },
  "bus_audit": {
    "accel_dvdt_corr": 0.9700520460458929,
    "steer_yaw_corr": 0.7271866055177758,
    "accel_sign": "+acceleration_x",
    "steering_sign_rule": "sign=1 -> negative; sign=0 -> positive"
  }
}


## 6. Final audit


In [8]:
import json, pandas as pd
from blackbox_detection.stage3.schema import read_frame_table

manifest = pd.read_csv(PROCESSED_ROOT / 'manifest.csv', dtype={'segment_id': str})
display(manifest)

print('segments:', len(manifest))
print('frames  :', int(manifest['num_frames'].sum()))
print('hours   :', float(manifest['num_frames'].sum() / 10 / 3600))

route = 'a2d2_20190401_121727'
for row in manifest.itertuples(index=False):
    sid = str(row.segment_id).zfill(3)
    video = PROCESSED_ROOT / row.video_relpath
    meta = PROCESSED_ROOT / row.metadata_relpath
    aux = PROCESSED_ROOT / row.aux_metadata_relpath
    done = PROCESSED_ROOT / 'done' / route / f'{sid}.json'
    assert video.is_file(), video
    assert meta.is_file(), meta
    assert aux.is_file(), aux
    assert done.is_file(), done
    df = read_frame_table(meta)
    assert len(df) == int(row.num_frames)
    assert df['frame_index_10hz'].tolist() == list(range(len(df)))

print('FINAL A2D2 PREP AUDIT: PASS')


,dataset,archive,vehicle_id,route_id,segment_id,num_frames,duration_s,video_relpath,metadata_relpath,aux_metadata_relpath
0,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,000,600,59.9,videos/a2d2_20190401_121727/000.mp4,metadata/a2d2_20190401_121727/000.npz,aux_metadata/a2d2_20190401_121727/000.npz
1,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,001,600,59.9,videos/a2d2_20190401_121727/001.mp4,metadata/a2d2_20190401_121727/001.npz,aux_metadata/a2d2_20190401_121727/001.npz
2,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,002,600,59.9,videos/a2d2_20190401_121727/002.mp4,metadata/a2d2_20190401_121727/002.npz,aux_metadata/a2d2_20190401_121727/002.npz
3,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,003,600,59.9,videos/a2d2_20190401_121727/003.mp4,metadata/a2d2_20190401_121727/003.npz,aux_metadata/a2d2_20190401_121727/003.npz
4,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,004,600,59.9,videos/a2d2_20190401_121727/004.mp4,metadata/a2d2_20190401_121727/004.npz,aux_metadata/a2d2_20190401_121727/004.npz
5,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,005,600,59.9,videos/a2d2_20190401_121727/005.mp4,metadata/a2d2_20190401_121727/005.npz,aux_metadata/a2d2_20190401_121727/005.npz
6,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,006,600,59.9,videos/a2d2_20190401_121727/006.mp4,metadata/a2d2_20190401_121727/006.npz,aux_metadata/a2d2_20190401_121727/006.npz
7,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,007,600,59.9,videos/a2d2_20190401_121727/007.mp4,metadata/a2d2_20190401_121727/007.npz,aux_metadata/a2d2_20190401_121727/007.npz
8,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,008,600,59.9,videos/a2d2_20190401_121727/008.mp4,metadata/a2d2_20190401_121727/008.npz,aux_metadata/a2d2_20190401_121727/008.npz
9,a2d2,camera_lidar-20190401121727_camera_frontcenter...,audi_a2d2,a2d2_20190401_121727,009,600,59.9,videos/a2d2_20190401_121727/009.mp4,metadata/a2d2_20190401_121727/009.npz,aux_metadata/a2d2_20190401_121727/009.npz


segments: 16
frames  : 9151
hours   : 0.25419444444444445
FINAL A2D2 PREP AUDIT: PASS


In [10]:
from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd

from blackbox_detection.stage3.schema import read_frame_table

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
PROCESSED_ROOT = DRIVE_ROOT / "DATASET/A2D2/processed"
ROUTE = "a2d2_20190401_121727"

MANIFEST = PROCESSED_ROOT / "manifest.csv"
REPORT = PROCESSED_ROOT / "prepare_report.json"
INDEX = PROCESSED_ROOT / "cache" / ROUTE / "camera_index.json"
INDEX_CKPT = (
    PROCESSED_ROOT
    / "cache"
    / ROUTE
    / "camera_index.checkpoint.json"
)
DONE_DIR = PROCESSED_ROOT / "done" / ROUTE

print("=== 1. CORE FILES ===")

for p in [MANIFEST, REPORT, INDEX]:
    print(p, "->", p.exists())
    assert p.is_file(), p

with open(REPORT, "r") as f:
    report = json.load(f)

with open(INDEX, "r") as f:
    camera_index = json.load(f)

manifest = pd.read_csv(
    MANIFEST,
    dtype={"segment_id": str},
)

print("\nindex complete:", camera_index.get("complete"))
assert camera_index.get("complete") is True

print("checkpoint remains:", INDEX_CKPT.exists())
# final index 완성 후 checkpoint는 없어지는 게 정상
assert not INDEX_CKPT.exists()


print("\n=== 2. GLOBAL COUNTS ===")

n_segments = len(manifest)
n_frames = int(manifest["num_frames"].sum())

print("manifest segments :", n_segments)
print("manifest frames   :", n_frames)
print("report segments   :", report["segments"])
print("report frames     :", report["selected_10hz_frames"])
print("duration minutes  :", n_frames / 10 / 60)

assert n_segments == int(report["segments"])
assert n_frames == int(report["selected_10hz_frames"])

# 이 세션은 약 15.3분이므로 대략 이 범위면 정상
assert 8500 <= n_frames <= 10000, n_frames
assert 14 <= n_segments <= 18, n_segments


print("\n=== 3. CAMERA INDEX ===")

camera_records = int(report["camera_records"])
index_records = len(camera_index["records"])
index_pngs = len(camera_index["png_members"])

print("camera JSON records:", camera_records)
print("indexed records    :", index_records)
print("indexed PNG        :", index_pngs)
print("members scanned    :", camera_index.get("members_scanned"))

assert camera_records == index_records
assert index_records > 25000
assert index_pngs > 25000

# 모든 JSON record가 대응 PNG를 가져야 함
png_names = set(camera_index["png_members"].keys())

missing_png = [
    r["png_member"]
    for r in camera_index["records"]
    if r["png_member"] not in png_names
]

print("missing paired PNG:", len(missing_png))
assert len(missing_png) == 0


print("\n=== 4. BUS / CAMERA ALIGNMENT ===")

bus_audit = report["bus_audit"]
skew = report["camera_skew_ms"]

print("accel dv/dt corr :", bus_audit["accel_dvdt_corr"])
print("steer-yaw corr   :", bus_audit["steer_yaw_corr"])
print("\n=== 4. BUS / CAMERA ALIGNMENT ===")

bus_audit = report["bus_audit"]
skew = report["camera_skew_ms"]

print("accel dv/dt corr :", bus_audit["accel_dvdt_corr"])
print("steer-yaw corr   :", bus_audit["steer_yaw_corr"])
print("camera skew p95  :", skew["p95_abs"], "ms")
print("camera skew max  :", skew["max_abs"], "ms")

assert bus_audit["accel_dvdt_corr"] > 0.80
assert bus_audit["steer_yaw_corr"] > 0.50
assert skew["max_abs"] <= 25.0

print("BUS / CAMERA ALIGNMENT: PASS")

assert bus_audit["accel_dvdt_corr"] > 0.80
assert bus_audit["steer_yaw_corr"] > 0.50
assert skew["max_abs"] <= 25.0


print("\n=== 5. SEGMENT-BY-SEGMENT AUDIT ===")

total_video_frames = 0
all_source_indices = []

for i, row in enumerate(manifest.itertuples(index=False)):
    seg_id = str(row.segment_id).zfill(3)

    video = PROCESSED_ROOT / row.video_relpath
    meta = PROCESSED_ROOT / row.metadata_relpath
    aux = PROCESSED_ROOT / row.aux_metadata_relpath
    done = DONE_DIR / f"{seg_id}.json"

    # files
    assert video.is_file(), video
    assert meta.is_file(), meta
    assert aux.is_file(), aux
    assert done.is_file(), done

    # video
    cap = cv2.VideoCapture(str(video))
    assert cap.isOpened(), video

    video_frames = int(round(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    video_fps = float(cap.get(cv2.CAP_PROP_FPS))
    cap.release()

    expected = int(row.num_frames)

    assert video_frames == expected, (
        seg_id, video_frames, expected
    )
    assert abs(video_fps - 10.0) < 0.2, (
        seg_id, video_fps
    )

    # shared metadata
    df = read_frame_table(meta)

    assert len(df) == expected
    assert df["frame_index_10hz"].tolist() == list(range(expected))
    assert df["timestamp"].is_monotonic_increasing
    assert not df["timestamp"].duplicated().any()

    assert (df["dataset"] == "a2d2").all()
    assert (df["route_id"] == ROUTE).all()
    assert (df["segment_id"].astype(str).str.zfill(3) == seg_id).all()

    # A2D2 raw steering wheel angle은 기존 comma steering regression에
    # 섞이지 않도록 valid_steer=False인 것이 의도된 동작
    assert not df["valid_steer"].astype(bool).any()

    # aux metadata
    with np.load(aux, allow_pickle=False) as a:
        for key in a.files:
            assert len(a[key]) == expected, (
                seg_id, key, len(a[key]), expected
            )

        required_aux = {
            "steering_wheel_deg",
            "steering_magnitude_deg",
            "steering_direction_sign",
            "acceleration_x_mps2",
            "brake_pressure_bar",
            "accelerator_pedal_pct",
        }

        assert required_aux.issubset(set(a.files)), (
            seg_id,
            required_aux - set(a.files),
        )

    # done marker
    with open(done, "r") as f:
        marker = json.load(f)

    assert int(marker["num_frames"]) == expected

    total_video_frames += video_frames
    all_source_indices.extend(
        df["frame_index_source"].astype(int).tolist()
    )

    print(
        f"{seg_id}: PASS | "
        f"frames={expected:4d} | "
        f"fps={video_fps:.2f} | "
        f"video={video.stat().st_size/1024**2:.1f} MiB"
    )


print("\n=== 6. CROSS-SEGMENT CHECK ===")

print("total video frames:", total_video_frames)
print("manifest frames   :", n_frames)

assert total_video_frames == n_frames

# 10Hz로 뽑은 원본 camera frame이 중복 선택되면 안 됨
print(
    "unique source indices:",
    len(set(all_source_indices)),
    "/",
    len(all_source_indices),
)

assert len(set(all_source_indices)) == len(all_source_indices)


print("\n=== 7. VALID TARGET COUNTS ===")

valid_cols = [
    "valid_speed",
    "valid_accel_from_speed",
    "valid_accel_imu",
    "valid_yaw",
]

counts = {c: 0 for c in valid_cols}

for row in manifest.itertuples(index=False):
    df = read_frame_table(PROCESSED_ROOT / row.metadata_relpath)
    for c in valid_cols:
        counts[c] += int(df[c].astype(bool).sum())

for k, v in counts.items():
    print(f"{k:24s}: {v:5d} / {n_frames}")

# 대부분의 프레임에 CAN supervision이 있어야 함
assert counts["valid_speed"] > n_frames * 0.95
assert counts["valid_accel_imu"] > n_frames * 0.95
assert counts["valid_yaw"] > n_frames * 0.95


print()
print("=" * 72)
print("FINAL A2D2 PREPROCESS AUDIT: PASS")
print(f"segments      : {n_segments}")
print(f"10 Hz frames  : {n_frames}")
print(f"duration      : {n_frames / 10 / 60:.2f} min")
print(f"camera records: {camera_records}")
print(f"max skew      : {skew['max_abs']:.3f} ms")
print("=" * 72)

=== 1. CORE FILES ===
/content/drive/MyDrive/Blackbox-Detection/DATASET/A2D2/processed/manifest.csv -> True
/content/drive/MyDrive/Blackbox-Detection/DATASET/A2D2/processed/prepare_report.json -> True
/content/drive/MyDrive/Blackbox-Detection/DATASET/A2D2/processed/cache/a2d2_20190401_121727/camera_index.json -> True

index complete: True
checkpoint remains: False

=== 2. GLOBAL COUNTS ===
manifest segments : 16
manifest frames   : 9151
report segments   : 16
report frames     : 9151
duration minutes  : 15.251666666666667

=== 3. CAMERA INDEX ===
camera JSON records: 27451
indexed records    : 27451
indexed PNG        : 27451
members scanned    : 54907
missing paired PNG: 0

=== 4. BUS / CAMERA ALIGNMENT ===
accel dv/dt corr : 0.9700520460458929
steer-yaw corr   : 0.7271866055177758

=== 4. BUS / CAMERA ALIGNMENT ===
accel dv/dt corr : 0.9700520460458929
steer-yaw corr   : 0.7271866055177758
camera skew p95  : 15.792 ms
camera skew max  : 16.669 ms
BUS / CAMERA ALIGNMENT: PASS

=== 5. 

## Resume 파일

```text
DATASET/A2D2/processed/
├── cache/a2d2_20190401_121727/
│   ├── camera_index.checkpoint.json   # scan 도중
│   └── camera_index.json              # scan 완료 후
├── done/a2d2_20190401_121727/
│   ├── 000.json
│   ├── 001.json
│   └── ...
├── videos/...
├── metadata/...
├── aux_metadata/...
├── manifest.csv
└── prepare_report.json
```
